# DSTS — LLM Inference Optimisation: 8B Model Benchmark (RunPod)

Runs the **vocabulary-pruning** experiments on `meta-llama/Llama-3.1-8B-Instruct`
on a **single** RunPod GPU pod (RTX 4090 24 GB recommended).

No Kaggle environment assumed — secrets are injected as `RUNPOD_SECRET_*` env vars.

## Prerequisites (do these ONCE before launching the pod)

### 1. HuggingFace token (for Llama-3.1-8B-Instruct access)
1. Go to https://huggingface.co/settings/tokens → create a token with **Read** scope
2. Accept the **Llama-3.1** license at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
   (separate gate from Llama-3.2 — must be accepted even if you already accepted 3.2)
3. RunPod → **Secrets** → **Add Secret**
   - Key: `HF_TOKEN`  Value: your token (`hf_...`)
4. In your pod template reference it as `{{ RUNPOD_SECRET_HF_TOKEN }}`

### 2. GitHub token (to clone the private repo)
1. GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic)
   → Generate new token → scope: **repo** (full)
2. RunPod → **Secrets** → **Add Secret**
   - Key: `GITHUB_TOKEN`  Value: your PAT (`ghp_...`)
3. In your pod template reference it as `{{ RUNPOD_SECRET_GITHUB_TOKEN }}`

### 3. GPU — single RTX 4090 (24 GB), RTX PRO 4500 Blackwell (32 GB), or A100 40 GB
`Llama-3.1-8B-Instruct` in **float16** requires ~16 GB VRAM; an RTX 4090 or
RTX PRO 4500 Blackwell fits it on a single GPU, giving clean `lm_head_frac`
measurements without multi-GPU sharding noise.
**Blackwell GPUs (SM 12.0):** Cell 1 auto-detects the mismatch with PyTorch < 2.7
and reinstalls PyTorch 2.7+ (cu128). The kernel restarts once — re-run from
Cell 1 and everything will work.

### 4. Pretrained artefacts
The `router_checkpoint.pt` and `mlp_transition_graph.pt` from the 1B run are
**incompatible** with the 8B model (hidden dim 2048 → 4096). Both are rebuilt
from scratch automatically (adds ~60 min). If you have 8B-compatible artefacts,
copy them to `/workspace/dsts/results/` before running Cell 5.

---
**After running:** download `results.zip` from `/workspace/results.zip` via the RunPod
file browser or `scp root@<pod-ip>:/workspace/results.zip .`  
**Locally:** `unzip results.zip -d dsts/results_8b/ && RESULTS_DIR=results_8b python run_plots.py`

In [ ]:
# ── Cell 1: Verify GPU + fix PyTorch/CUDA arch mismatch ─────────────────────────
#
# Blackwell GPUs (RTX PRO 4500, RTX 5090, etc. — SM 12.0) require PyTorch ≥ 2.7.
# Older PyTorch wheels lack SM 12.0 cubins and crash with:
#   RuntimeError: CUDA error: no kernel image is available for execution on the device
# This cell detects the mismatch and reinstalls the correct wheel automatically.
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — check pod GPU settings')

import torch
print(f'torch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        major, minor = torch.cuda.get_device_capability(i)
        print(f'CUDA device {i}: {name}, {vram:.1f} GB  (SM {major}.{minor})')

# ── Detect SM 12.0+ (Blackwell) with a PyTorch build that predates it ──────────
needs_reinstall = False
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    sm = major * 10 + minor
    torch_major = int(torch.__version__.split('.')[0])
    torch_minor = int(torch.__version__.split('.')[1])
    if sm >= 120 and (torch_major, torch_minor) < (2, 7):
        needs_reinstall = True
        print(f'\n*** SM {major}.{minor} (Blackwell) detected with PyTorch {torch.__version__} ***')
        print('*** PyTorch < 2.7 has no SM 12.0 cubin — reinstalling PyTorch 2.7+ (cu128)... ***\n')

import pathlib
SENTINEL = pathlib.Path('/tmp/.torch_reinstalled')

if SENTINEL.exists():
    # ── Second run after manual kernel restart ──────────────────────────────
    SENTINEL.unlink()
    import torch
    print(f'Kernel restarted successfully. PyTorch {torch.__version__} loaded.')
    print('PyTorch/CUDA arch OK — continue with Cell 2.')
elif needs_reinstall:
    # --force-reinstall overwrites the system torch in-place so the kernel
    # restart picks up the new binaries from the same path.
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--force-reinstall', '--upgrade',
        'torch', 'torchvision', 'torchaudio',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
    ])
    # Confirm the installed version before restarting
    r = subprocess.run([sys.executable, '-m', 'pip', 'show', 'torch'],
                       capture_output=True, text=True)
    version_line = next((l for l in r.stdout.splitlines() if l.startswith('Version')), 'Version: unknown')
    print(f'pip show torch → {version_line}')
    SENTINEL.touch()
    print()
    print('=' * 60)
    print('PyTorch 2.7+cu128 force-reinstalled.')
    print()
    print('  *** RESTART THE KERNEL NOW ***')
    print('  Kernel menu → Restart Kernel  (or click the restart button)')
    print('  Then re-run Cell 1 once — it will confirm success.')
    print('=' * 60)
    print('PyTorch 2.7+cu128 installed.')
    print()
    print('  *** RESTART THE KERNEL NOW ***')
    print('  Kernel menu → Restart Kernel  (or click the restart button)')
    print('  Then re-run Cell 1 once — it will confirm success.')
    print('=' * 60)
else:
    print('PyTorch/CUDA arch OK — no reinstall needed.')

try:
    import triton
    print(f'Triton: {triton.__version__} ✓  (fast router kernel available)')
except ImportError:
    print('Triton: not installed — PyTorch fast-path will be used instead')

In [ ]:
# ── Cell 2: HuggingFace token + model selection ───────────────────────────────
# RunPod injects secrets as env vars with the RUNPOD_SECRET_ prefix.
# Reference them in your pod template as {{ RUNPOD_SECRET_HF_TOKEN }}.
import os

hf_token = os.environ.get('RUNPOD_SECRET_HF_TOKEN') or os.environ.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError(
        'HF_TOKEN not found.\n'
        'Set it in RunPod Secrets as HF_TOKEN and reference it in the pod template '
        'as {{ RUNPOD_SECRET_HF_TOKEN }}, or set HF_TOKEN directly in the environment.'
    )

os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token  # legacy env var
print('HF_TOKEN loaded ✓')

# Select the 8B model — all run_*.py scripts read MODEL_NAME from the environment
os.environ['MODEL_NAME'] = 'meta-llama/Llama-3.1-8B-Instruct'
print(f'MODEL_NAME set to: {os.environ["MODEL_NAME"]}')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────────
# RunPod PyTorch images ship with torch/CUDA; install the remaining packages.
# triton is included with recent PyTorch — checked in Cell 1.
# faiss is intentionally omitted: ANN search uses a plain PyTorch matmul (N=8000).
!pip install -q \
    'transformers>=4.40.0' \
    'accelerate>=0.29.0' \
    'datasets>=2.18.0' \
    'sentencepiece>=0.2.0' \
    'tokenizers>=0.19.0' \
    'scikit-learn>=1.4.0' \
    'sentence-transformers>=2.7.0' \
    'tqdm>=4.66.0' \
    'numpy>=1.26.0' \
    'pandas>=2.2.0' \
    'matplotlib>=3.8.0' \
    'seaborn>=0.13.0'

print('Dependencies installed ✓')

In [ ]:
# ── Cell 4: Clone repo (private) ─────────────────────────────────────────────
# Reads GITHUB_TOKEN from the RunPod secret injected as RUNPOD_SECRET_GITHUB_TOKEN.
import os

gh_token = os.environ.get('RUNPOD_SECRET_GITHUB_TOKEN') or os.environ.get('GITHUB_TOKEN')
if not gh_token:
    raise RuntimeError(
        'GITHUB_TOKEN not found.\n'
        'Set it in RunPod Secrets as GITHUB_TOKEN and reference it in the pod template '
        'as {{ RUNPOD_SECRET_GITHUB_TOKEN }}, or set GITHUB_TOKEN directly in the environment.'
    )

WORKDIR = '/workspace/dsts'
REPO_URL = f'https://{gh_token}@github.com/deil87/dsts.git'

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {WORKDIR} remote set-url origin {REPO_URL}
    !git -C {WORKDIR} pull

%cd {WORKDIR}
print(f'Working directory: {os.getcwd()}')

# Re-export after %cd (subshells inherit env from the kernel)
os.environ['MODEL_NAME'] = 'meta-llama/Llama-3.1-8B-Instruct'
print(f'MODEL_NAME: {os.environ["MODEL_NAME"]}')
!ls -la

In [ ]:
# ── Cell 5: Artefacts setup + archive old DE results ───────────────────────────
# If you have 8B-compatible artefacts (router_checkpoint.pt trained with hidden_dim=4096),
# copy them to /workspace/dsts/results/ before running this cell.
# 1B artefacts (hidden_dim=2048) are INCOMPATIBLE and will be silently ignored.
#
# IMPORTANT: any existing dual_encoder_results.csv / dual_encoder_prefetch_results.csv
# are archived to results/old_slow_router/ so that run_dual_encoder.py runs fresh
# with the GPU-native fast router (Triton kernel + zero CPU round-trips).
import os, shutil

os.makedirs('results', exist_ok=True)

# ── Check artefacts ──
for fname in ['router_checkpoint.pt', 'mlp_transition_graph.pt']:
    path = f'results/{fname}'
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'Found artefact: {path} ({size_mb:.1f} MB)')
        print('  WARNING: verify this is an 8B checkpoint (hidden_dim=4096); 1B checkpoints are incompatible.')
    else:
        print(f'No artefact at {path} — will build from scratch (expected).')

# ── Archive ALL stale result CSVs so every experiment runs fresh ──
# This prevents sections from being skipped due to stale results committed
# in the repo from previous runs on different hardware.
ALL_RESULT_FILES = [
    'results/baseline_summary.csv',
    'results/static_router_results.csv',
    'results/cosine_router_results.csv',
    'results/cluster_router_results.csv',
    'results/graph_router_results.csv',
    'results/dual_encoder_results.csv',
    'results/dual_encoder_prefetch_results.csv',
    'results/shortlist_sizes.npy',
    'results/all_results.csv',
]
old_dir = 'results/old_slow_router'
archived = []
for p in ALL_RESULT_FILES:
    if os.path.exists(p):
        os.makedirs(old_dir, exist_ok=True)
        dest = os.path.join(old_dir, os.path.basename(p))
        shutil.move(p, dest)
        archived.append(f'  {p} → {dest}')

if archived:
    print('\nArchived stale results (will run all experiments fresh):')
    for line in archived:
        print(line)
else:
    print('\nNo stale results found — will run fresh from scratch.')

In [ ]:
# ── Cell 6: Run baseline ────────────────────────────────────────────────────────
# Expected runtime: ~8 min (8B on single RTX 4090)
print('=' * 60)
print('EXPERIMENT 1/6: Baseline')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_baseline.py

In [ ]:
# ── Cell 7: Static + Cosine router ──────────────────────────────────────────────
# Expected runtime: ~30 min (8B on single RTX 4090)
print('=' * 60)
print('EXPERIMENT 2/6: Static + Cosine Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_static.py

In [ ]:
# ── Cell 8: Cluster router ──────────────────────────────────────────────────────
# Expected runtime: ~20 min (8B on single RTX 4090)
print('=' * 60)
print('EXPERIMENT 3/6: Cluster Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_cluster.py

In [ ]:
# ── Cell 9: Dual-encoder router ───────────────────────────────────────────────
# Trains RouterMLP (if 8B checkpoint not found) then runs step-level + prefetch/refresh.
# Expected runtime: ~90 min (8B on single RTX 4090, includes ~50 min training)
# Fast path: Triton-fused MLP + GPU-native token union (no CPU round-trips).
print('=' * 60)
print('EXPERIMENT 4/6: Dual-Encoder Router (fast path)')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_dual_encoder.py

In [ ]:
# ── Cell 10: MLP transition graph router ────────────────────────────────────────
# Expected runtime: ~45 min (8B on single RTX 4090, includes ~15 min graph build)
print('=' * 60)
print('EXPERIMENT 5/6: MLP Transition Graph Router')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_graph.py

In [ ]:
# ── Cell 11: Plots ──────────────────────────────────────────────────────────────
# Expected runtime: ~1 min
print('=' * 60)
print('EXPERIMENT 6/6: Generate Plots')
print('=' * 60)
!MODEL_NAME=meta-llama/Llama-3.1-8B-Instruct python run_plots.py

In [ ]:
# ── Cell 12: Summary of results ───────────────────────────────────────────────
import pandas as pd
import os

print('Results directory contents:')
for f in sorted(os.listdir('results')):
    fpath = f'results/{f}'
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f'  {f:<50s} {size/1024:>8.1f} KB')

# Print the consolidated results table if it exists
all_csv = 'results/all_results.csv'
if os.path.exists(all_csv):
    print('\n=== All Results ===')
    df = pd.read_csv(all_csv)
    with pd.option_context('display.max_rows', 50, 'display.max_columns', 20,
                           'display.width', 120, 'display.float_format', '{:.3f}'.format):
        print(df.to_string(index=False))

In [ ]:
# ── Cell 13: Package results for download ─────────────────────────────────────
# Zips the entire results/ directory → /workspace/results.zip
# Download via the RunPod file browser, or:
#   scp root@<pod-ip>:/workspace/results.zip .
#   rsync -avz root@<pod-ip>:/workspace/results.zip .
import shutil, os

OUTPUT_ZIP = '/workspace/results'
shutil.make_archive(OUTPUT_ZIP, 'zip', 'results')

zip_size = os.path.getsize(OUTPUT_ZIP + '.zip') / 1e6
print(f'results.zip created: {zip_size:.1f} MB  →  /workspace/results.zip')
print()
print('To download from your local machine:')
print('  scp root@<pod-ip>:/workspace/results.zip .')
print()
print('Locally, unzip into your dsts/results_8b/ folder:')
print('  mkdir -p results_8b')
print('  unzip results.zip -d results_8b/')
print('  RESULTS_DIR=results_8b python run_plots.py')